<a href="https://colab.research.google.com/github/nacoajaree/datastructure/blob/main/calVal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
def infix_to_postfix(expression):
    # 1. กำหนดลำดับความสำคัญของเครื่องหมาย (ยิ่งเลขมาก ยิ่งสำคัญมาก)
    precedence = {'+': 1, '-': 1, '*': 2, '/': 2, '^': 3}

    stack = []       # ตัวเก็บเครื่องหมายชั่วคราว
    output = []      # ตัวเก็บผลลัพธ์ที่จะได้เป็น Postfix

    # 2. แยกข้อความออกเป็นตัวๆ (ลบช่องว่างออกก่อน)
    # สมมติว่าอินพุตเป็นตัวอักษรติดกัน เช่น A+B*(C-D)
    for char in expression.replace(" ", ""):

        # ถ้าเป็นตัวอักษรหรือตัวเลข (Operand) ให้ใส่ใน output ทันที
        if char.isalnum():
            output.append(char)

        # ถ้าเป็นวงเล็บเปิด '(' ให้ใส่เข้าไปใน stack
        elif char == '(':
            stack.append(char)

        # ถ้าเป็นวงเล็บปิด ')' ให้ดึงเครื่องหมายใน stack ออกมาใส่ output จนกว่าจะเจอ '('
        elif char == ')':
            while stack and stack[-1] != '(':
                output.append(stack.pop())
            stack.pop() # เอาวงเล็บเปิด '(' ออกจาก stack ไปด้วย

        # ถ้าเป็นตัวดำเนินการ (+, -, *, /, ^)
        else:
            # ตราบใดที่เครื่องหมายบนสุดของ stack มีความสำคัญมากกว่าหรือเท่ากับตัวปัจจุบัน
            # ให้ดึงเครื่องหมายบนสุดของ stack ออกมาใส่ใน output ก่อน
            while stack and stack[-1] != '(' and precedence.get(stack[-1], 0) >= precedence.get(char, 0):
                output.append(stack.pop())
            # เสร็จแล้วค่อยใส่ตัวดำเนินการปัจจุบันลงไปใน stack
            stack.append(char)

    # 3. ถ้าอ่านนิพจน์จนจบแล้วยังมีเครื่องหมายเหลืออยู่ใน stack ให้ดึงออกมาใส่ output ให้หมด
    while stack:
        output.append(stack.pop())

    # รวมลิสต์ผลลัพธ์กลับมาเป็นข้อความตัวยาวๆ
    return "".join(output)

# --- ทดลองใช้งาน ---
if __name__ == "__main__":
    # ตัวอย่างที่ 1: ไม่มีวงเล็บ (ทดสอบลำดับความสำคัญ คูณหารต้องทำก่อนบวกลบ)
    expr1 = "A + B * C"
    print(f"Infix  : {expr1}")
    print(f"Postfix: {infix_to_postfix(expr1)}")  # ผลลัพธ์ควรได้: ABC*+

    print("-" * 30)

    # ตัวอย่างที่ 2: มีวงเล็บ (ทดสอบการบังคับทำในวงเล็บก่อน)
    expr2 = "(A + B) * (C - D)"
    print(f"Infix  : {expr2}")
    print(f"Postfix: {infix_to_postfix(expr2)}")  # ผลลัพธ์ควรได้: AB+CD-*

    print("-" * 30)

    # ตัวอย่างที่ 3: ทดสอบการดำเนินการซ้ายไปขวา
    expr3 = "A - B + C * (D / F)"
    print(f"Infix  : {expr3}")
    print(f"Postfix: {infix_to_postfix(expr3)}")  # ผลลัพธ์ควรได้:

     # ตัวอย่างที่ 4: ทดสอบการยกกำลัง
    expr4 = "A ^ B ^ C + D * F"
    print(f"Infix  : {expr4}")
    print(f"Postfix: {infix_to_postfix(expr4)}")  # ผลลัพธ์ควรได้:

Infix  : A + B * C
Postfix: ABC*+
------------------------------
Infix  : (A + B) * (C - D)
Postfix: AB+CD-*
------------------------------
Infix  : A - B + C * (D / F)
Postfix: AB-CDF/*+
Infix  : A ^ B ^ C + D * F
Postfix: AB^C^DF*+


In [12]:
def evaluate_postfix(expression):
    stack = []

    # แยกส่วนประกอบด้วยช่องว่าง เพื่อรองรับตัวเลขหลายหลัก (เช่น 10, 25)
    tokens = expression.split()

    for token in tokens:
        # 1. ถ้าเป็นตัวเลข (ตรวจเช็กว่าเป็นเลขจำนวนเต็ม หรือเลขติดลบได้)
        if token.isdigit() or (token.startswith('-') and token[1:].isdigit()):
            stack.append(int(token))

        # 2. ถ้าเป็นเครื่องหมายตัวดำเนินการ
        else:
            # ดึงตัวเลข 2 ตัวล่าสุดออกจากสแตก
            # ตัวที่ป็อปออกมาก่อนจะเป็นตัวตั้งขวา (operand2) ตัวที่ออกทีหลังจะเป็นตัวตั้งซ้าย (operand1)
            operand2 = stack.pop()
            operand1 = stack.pop()

            # คำนวณตามเครื่องหมาย
            if token == '+':
                result = operand1 + operand2
            elif token == '-':
                result = operand1 - operand2
            elif token == '*':
                result = operand1 * operand2
            elif token == '/':
                # ใช้ // สำหรับการหารเอาเศษ หรือเปลี่ยนเป็น / ถ้าต้องการทศนิยม
                result = operand1 // operand2
            elif token == '^':
                result = operand1 ** operand2

            # นำผลลัพธ์ที่ได้ใส่กลับเข้าไปในสแตก
            stack.append(result)

    # คำตอบสุดท้ายจะเหลืออยู่ตัวเดียวในสแตก
    return stack.pop()

# --- ทดลองใช้งาน ---
if __name__ == "__main__":
    # ตัวอย่างที่ 1: "2 3 *" หมายถึง 2 * 3 = 6
    # จากนั้น "6 4 +" หมายถึง 6 + 4 = 10
    expr1 = "2 3 * 4 +"
    print(f"Postfix   : {expr1}")
    print(f"Result (10): {evaluate_postfix(expr1)}")

    print("-" * 30)

    # ตัวอย่างที่ 2: มาจาก infix: (15 + 5) * (10 - 8)
    # แปลงเป็น postfix ได้: 15 5 + 10 8 - *
    # คำนวณ: (20) * (2) = 40
    expr2 = "15 5 + 10 8 - *"
    print(f"Postfix   : {expr2}")
    print(f"Result (40): {evaluate_postfix(expr2)}")

Postfix   : 2 3 * 4 +
Result (10): 10
------------------------------
Postfix   : 15 5 + 10 8 - *
Result (40): 40
